# 2. Graph Extraction (Pure GraphRAG)

Notebook ini mengekstrak entitas dan relasi dari raw EMR menggunakan LLM, lalu menyimpannya ke Neo4j.

In [ ]:
import os
import sys

# --- Robust project-root detection (works regardless of CWD or kernel) ---
_cwd = os.path.abspath(os.getcwd())
_project_root = None
_temp = _cwd
for _ in range(5):
    if os.path.exists(os.path.join(_temp, "src", "config.py")):
        _project_root = _temp
        break
    _parent = os.path.abspath(os.path.join(_temp, ".."))
    if _parent == _temp:
        break
    _temp = _parent

if not _project_root:
    # Fallback: scan sys.path entries
    for _p in list(sys.path):
        if not _p:
            continue
        for _candidate in [_p, os.path.join(_p, ".."), os.path.join(_p, "..", "..")]:
            _candidate = os.path.abspath(_candidate)
            if os.path.exists(os.path.join(_candidate, "src", "config.py")):
                _project_root = _candidate
                break
        if _project_root:
            break

if _project_root:
    if _project_root not in sys.path:
        sys.path.insert(0, _project_root)
    print(f"Project root: {_project_root}")
else:
    raise RuntimeError(
        "Cannot locate project root. "
        "Make sure you are running this notebook from inside the project folder."
    )

from src.config import settings

import pandas as pd
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

from src.graph.client import GraphClient
from src.ingestion.extractor import LLMGraphExtractor
from src.services.providers import get_embeddings

import logging
logging.basicConfig(level=logging.INFO)

In [ ]:
# ============================================================
# Step 1.5: Setup Neo4j Indexes (CRITICAL)
# Must run before extracting nodes to ensure fast merges and vector search capability.
# ============================================================
print('Setting up Neo4j Indexes... ')
try:
    from scripts.setup_indexes import main as run_setup_indexes
    run_setup_indexes()
    print('  [OK] setup_indexes done')
except Exception as e:
    print(f'  [WARN] setup_indexes failed: {e}')


In [ ]:
client = GraphClient(
    uri=settings.neo4j_uri,
    user=settings.neo4j_user,
    password=settings.neo4j_password
)
extractor = LLMGraphExtractor(temperature=0.0)

In [ ]:
import os
_data_file = os.path.join(_project_root, "data", "Dashboard EMR(report1776669858353).csv")
try:
    df = pd.read_csv(_data_file, encoding="utf-8-sig")
except UnicodeDecodeError:
    try:
        df = pd.read_csv(_data_file, encoding="latin1")
    except Exception:
        df = pd.read_csv(_data_file, encoding="cp1252")

df_sample = df  # Jalankan semua data
print(f"Loaded {len(df_sample):,} records from: {_data_file}")

In [ ]:
def ingest_extraction_to_neo4j(extraction, emr_name, client):
    if not extraction: return
    client.run_query("MERGE (e:EMRRecord {emr_name: $name})", {"name": emr_name})
    for node in extraction.nodes:
        query = f"""
        MERGE (n:{node.label} {{name: $name}})
        WITH n MATCH (e:EMRRecord {{emr_name: $emr_name}})
        MERGE (e)-[:MENTIONS]->(n)
        """
        client.run_query(query, {"name": node.name, "emr_name": emr_name})
    for rel in extraction.relationships:
        rel_query = f"""
        MATCH (s {{name: $source}}) MATCH (t {{name: $target}}) MERGE (s)-[:{rel.type}]->(t)
        """
        try: client.run_query(rel_query, {"source": rel.source, "target": rel.target})
        except Exception: pass

In [ ]:
start_time = time.time()
existing = client.run_query("MATCH (e:EMRRecord) RETURN e.emr_name AS name")
existing_names = set(record['name'] for record in existing)
print(f"Found {len(existing_names)} already processed records. Skipping...")

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

records_to_process = []
for idx, row in df_sample.iterrows():
    emr_name = str(row.get('EMR Name', row.get('emr_name', f'EMR-UNK-{idx}'))).strip()
    if not emr_name or emr_name.lower() == 'nan':
        continue
    if emr_name in existing_names:
        continue
    text_chunk = f"""
    EMR Record ID: {emr_name}\nMachine Model: {row.get('Machine Model', '')}\nComponent: {row.get('Techcare Component', '')} | Sub-Component: {row.get('Techcare Sub Component', '')}\nSymptom/Problem: {row.get('Symptom', '')}\nCause of Problem: {row.get('Caused of Problem', '')}\nAction Taken: {row.get('Action (How was problem corrected?)', '')}\nPart Replaced: {row.get('Part Description', '')} (Part No: {row.get('Main Cause Part No', '')})\n
    """
    records_to_process.append((emr_name, text_chunk))

total = len(records_to_process)
counter = [0]
lock = threading.Lock()
print(f"Processing {total} new records with 8 parallel workers...")

def process_one(emr_name, text_chunk):
    import time as _time
    for attempt in range(3):
        try:
            extraction = extractor.extract(text_chunk)
            if extraction:
                ingest_extraction_to_neo4j(extraction, emr_name, client)
            with lock:
                counter[0] += 1
                done = counter[0]
            if done % 100 == 0:
                print(f"  Progress: {done}/{total}")
            return True
        except Exception as e:
            err_str = str(e)
            if '429' in err_str or 'RateLimit' in err_str or 'TooManyRequests' in err_str:
                wait = (2 ** attempt) + 0.5
                print(f"  Rate limited on {emr_name}, retrying in {wait:.1f}s...")
                _time.sleep(wait)
            else:
                if attempt == 0:
                    print(f"Error extracting {emr_name}: {e}")
                return False
    print(f"Failed {emr_name} after 3 retries")
    return False

with ThreadPoolExecutor(max_workers=8) as pool:
    futures = [pool.submit(process_one, emr_name, text_chunk) for emr_name, text_chunk in records_to_process]
    for f in as_completed(futures):
        pass

print(f"""\nCompleted in {time.time() - start_time:.2f} seconds.""")


## Generate Embeddings untuk Entity Resolution
Agar kita bisa melakukan deteksi kesamaan pada *Notebook 3*, setiap entitas harus memiliki representasi *vector embedding*.

In [ ]:
from src.graph.embedding_writer import BatchEmbeddingWriter

print("Men-generate Vector Embeddings (batch mode)...")
embedder = get_embeddings()
writer = BatchEmbeddingWriter(client, batch_size=500)

labels_to_embed = ["SymptomPattern", "ProblemCluster", "RootCausePattern", "ActionPattern"]

for label in labels_to_embed:
    nodes = client.run_query(f"MATCH (n:{label}) WHERE n.embedding IS NULL RETURN n.name AS name")
    names = [n['name'] for n in nodes if n['name']]
    if names:
        print(f"Embedding {len(names)} {label}...")
        embeddings = embedder.embed_documents(names)
        updated = writer.write_embeddings(label, names, embeddings)
        print(f"  → {updated} nodes updated")

print("Embeddings selesai!")


## Deterministic Enrichment (Post-Processing)
Langkah ini memastikan bahwa seluruh **Techcare Component** dan **ActionPattern** dari CSV disuntikkan secara deterministik dan terhubung dengan benar ke graf, menutupi kelemahan LLM yang mungkin melewatkan entitas tersebut.

In [ ]:
from src.ingestion.graph_enricher import GraphEnricher
import time

print("Memulai deterministic enrichment (batch mode)...")
start_time = time.time()

enricher = GraphEnricher(client, batch_size=500)
enricher.enrich(df_sample)

print(f"Enrichment selesai dalam {time.time() - start_time:.1f}s.")


In [ ]:
client.close()